In [ ]:
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
INPUT_PATH = "Broken_terrains_datasets"
REAL_DATA_FILE = "KSH_input_output_0.txt"

RANDOM_STATE = 42

ARTIFACTS_DIR = Path("artifacts/")
if not ARTIFACTS_DIR.exists():
    ARTIFACTS_DIR = Path("..") / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
pd.set_option('display.max_columns', None)

np.random.seed(RANDOM_STATE)
random_state = np.random.RandomState(RANDOM_STATE)

In [ ]:
files = os.listdir(INPUT_PATH)
dfs = [pd.read_csv(os.path.join(INPUT_PATH, str(file) + ".txt"), decimal='.', sep=';') for file in range(1000)]

In [ ]:
df = pd.concat(
    dfs,
    ignore_index=True
)

In [ ]:
filtered_df=df[
    (df.X_C_Neighbor1!='undefined') 
    & (df.X_C_Neighbor2!='undefined')
    & (df.X_C_Neighbor3!='undefined') 
    & (df.Z_N!=0) 
    & (df.n1_zn!=0) 
    & (df.n2_zn!=0) 
    & (df.n3_zn!=0) 
    & (df.DOC<0.90)  
].reset_index(drop=True)

In [ ]:
euclidean_n = ['EuclideanNeighbor1_N', 'EuclideanNeighbor2_N','EuclideanNeighbor3_N']
euclidean_d = ['EuclideanNeighbor1_D', 'EuclideanNeighbor2_D','EuclideanNeighbor3_D']
cosine_n = ['CosineNeighbor1_N', 'CosineNeighbor2_N','CosineNeighbor3_N']
cosine_d = ['CosineNeighbor1_D', 'CosineNeighbor2_D','CosineNeighbor3_D']
angle_n = ['AngleNeighbor1_N', 'AngleNeighbor2_N','AngleNeighbor3_N']
angle_d = ['AngleNeighbor1_D', 'AngleNeighbor2_D','AngleNeighbor3_D']

euclidean_n_sorted = ['Euclidean_N_Max', 'Euclidean_N_Min', 'Euclidean_N_Intermediate']
euclidean_d_sorted = ['Euclidean_D_Max', 'Euclidean_D_Min', 'Euclidean_D_Intermediate']
cosine_n_sorted = ['Cosine_N_Max', 'Cosine_N_Min', 'Cosine_N_Intermediate']
cosine_d_sorted = ['Cosine_D_Max', 'Cosine_D_Min', 'Cosine_D_Intermediate']
angle_n_sorted = ['Angle_N_Max', 'Angle_N_Min', 'Angle_N_Intermediate']
angle_d_sorted = ['Angle_D_Max', 'Angle_D_Min', 'Angle_D_Intermediate']

sorting_pairs = [
    (euclidean_n, euclidean_n_sorted),
    (euclidean_d, euclidean_d_sorted),
    (cosine_n, cosine_n_sorted),
    (cosine_d, cosine_d_sorted),
    (angle_n, angle_n_sorted),
    (angle_d, angle_d_sorted)
]

In [ ]:
def sort_values(row: pd.Series, output_columns: list) -> pd.Series:
    """
    Sort Neighbor values in descending order and return a Series with max, intermediate, and min values.

    Parameters
    ----------
    row : pd.Series
        A pandas Series containing Neighbor values.
    output_columns : list
        A list of column names for the output Series.
        Maximum value, intermediate value, minimum value.

    Returns
    -------
    pd.Series
        A pandas Series with the maximum, intermediate, and minimum values.
    """
    max_val = row.max()
    min_val = row.min()
    remaining_val = row.sum() - max_val - min_val
    return pd.Series([max_val, min_val, remaining_val], index=output_columns)


In [ ]:
sorted_dfs = [
    filtered_df[list(cols)].apply(sort_values, axis=1, output_columns=list(sorted_cols))
    for cols, sorted_cols in sorting_pairs
]

In [ ]:
sorted_df=pd.concat([
    filtered_df[['X_N']],
    filtered_df[['Y_N']],
    filtered_df[['Z_N']],
    filtered_df[['X_D']],
    filtered_df[['Y_D']],
    filtered_df[['Z_D']],   
    *sorted_dfs,
    filtered_df[['File_number']],
    filtered_df[['Fault']]   
    ], 
    axis=1
)
sorted_df.shape

## Real surface - same filter + neighbor sort


In [ ]:
real_raw_df = pd.read_csv(INPUT_PATH + "/" + REAL_DATA_FILE, decimal=".", sep=";")
real_raw_df.shape

In [ ]:
real_filtered_df=real_raw_df[
    (real_raw_df.X_C_Neighbor1!='undefined') 
    & (real_raw_df.X_C_Neighbor2!='undefined')
    & (real_raw_df.X_C_Neighbor3!='undefined') 
    & (real_raw_df.Z_N!=0) 
    & (real_raw_df.n1_zn!=0) 
    & (real_raw_df.n2_zn!=0) 
    & (real_raw_df.n3_zn!=0) 
    & (real_raw_df.DOC<0.90)  
].reset_index(drop=True)
real_filtered_df.shape

In [ ]:
real_sorted_dfs = [
    real_filtered_df[list(cols)].apply(sort_values, axis=1, output_columns=list(sorted_cols))
    for cols, sorted_cols in sorting_pairs
]

In [ ]:
real_sorted_df=pd.concat([
    real_filtered_df[["X_C","Y_C","Z_C"]],
    real_filtered_df[['X_N']],
    real_filtered_df[['Y_N']],
    real_filtered_df[['Z_N']],
    real_filtered_df[['X_D']],
    real_filtered_df[['Y_D']],
    real_filtered_df[['Z_D']],   
    *real_sorted_dfs,
    ], 
    axis=1
)
real_sorted_df.shape

## Synthetic parameter table from `params.txt`


In [ ]:
params_path = Path(INPUT_PATH) / "params.txt"
params_text = params_path.read_text(encoding="utf-8", errors="replace")

def _float_after(label: str) -> float | None:
    m = re.search(rf"{re.escape(label)}\s*:?\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", params_text)
    return float(m.group(1)) if m else None

n_files_m = re.search(r"Number of files:\s*(\d+)", params_text)
orient_rows = re.findall(
    r"For the (\d+)-file we have:\s*Dip ang:\s*([-+\d.eE]+)\.\s*Dip direction:\s*([-+\d.eE]+)\.",
    params_text,
)
orient_df = pd.DataFrame(
    [(int(i), float(d), float(az)) for i, d, az in orient_rows],
    columns=["file", "dip_ang", "dip_direction"],
)

# Class proportions on filtered synthetic observations (Fault: -1 homocline, 1 fault)
fault_counts = filtered_df["Fault"].value_counts(dropna=False)
n_obs = int(len(filtered_df))
n_fault = int(fault_counts.get(1, 0))
n_hom = int(fault_counts.get(-1, 0))

params_table = pd.DataFrame(
    [
        {"parameter": "n_synthetic_surfaces", "value": int(n_files_m.group(1)) if n_files_m else len(dfs)},
        {"parameter": "left_terrain_size", "value": _float_after("Left terrain size")},
        {"parameter": "right_terrain_size", "value": _float_after("Right terrain size")},
        {"parameter": "azimuth_range_left", "value": _float_after("Left range azimuth given")},
        {"parameter": "azimuth_range_right", "value": _float_after("Right range azimuth given")},
        {"parameter": "minimum_dip", "value": _float_after("Minimum dip given")},
        {"parameter": "maximum_dip", "value": _float_after("Maximum dip terrain given")},
        {"parameter": "triangulation_points_lower", "value": _float_after("Lower bound for points in the triangulation")},
        {"parameter": "triangulation_points_upper", "value": _float_after("Upper bound for points in the triangulation")},
        {"parameter": "noise_lower", "value": _float_after("Lower bound for noise")},
        {"parameter": "noise_upper", "value": _float_after("Upper bound for noise")},
        {"parameter": "fault_throw_lower", "value": _float_after("Lower bound for fault throw")},
        {"parameter": "fault_throw_upper", "value": _float_after("Upper bound for fault throw")},
        {"parameter": "n_filtered_synth_observations", "value": n_obs},
        {"parameter": "class_homocline_count_Fault_-1", "value": n_hom},
        {"parameter": "class_fault_count_Fault_1", "value": n_fault},
        {"parameter": "class_fault_proportion", "value": round(n_fault / n_obs, 4) if n_obs else None},
        {"parameter": "mean_dip_ang_from_params", "value": round(float(orient_df.dip_ang.mean()), 4) if len(orient_df) else None},
        {"parameter": "mean_dip_direction_from_params", "value": round(float(orient_df.dip_direction.mean()), 4) if len(orient_df) else None},
    ]
)
params_table.to_csv(ARTIFACTS_DIR / "synthetic_parameters_table.csv", index=False)
orient_df.to_csv(ARTIFACTS_DIR / "synthetic_per_file_orientations.csv", index=False)
params_table

assert len(orient_df) == 1000


## Surface-level feature comparison

Compares **every** modelling feature used by the modelling notebooks.

In [ ]:
# All modeling columns shared by synth sorted_df and real_sorted_df
feature_cols = [
    "X_N", "Y_N", "Z_N", "X_D", "Y_D", "Z_D",
    "Euclidean_N_Max", "Euclidean_N_Min", "Euclidean_N_Intermediate",
    "Euclidean_D_Max", "Euclidean_D_Min", "Euclidean_D_Intermediate",
    "Cosine_N_Max", "Cosine_N_Min", "Cosine_N_Intermediate",
    "Cosine_D_Max", "Cosine_D_Min", "Cosine_D_Intermediate",
    "Angle_N_Max", "Angle_N_Min", "Angle_N_Intermediate",
    "Angle_D_Max", "Angle_D_Min", "Angle_D_Intermediate",
]

COLOR_SURFACE = "#4c78a8"
COLOR_REAL = "#e45756"
COLOR_SYNTH_GLOBAL = "#f58518"

synth_X = sorted_df[feature_cols].apply(pd.to_numeric, errors="coerce")
real_X_geom = real_sorted_df[feature_cols].apply(pd.to_numeric, errors="coerce")

# Per-surface means (synth) + markers
surface_means = (
    sorted_df.assign(**{c: pd.to_numeric(sorted_df[c], errors="coerce") for c in feature_cols})
    .groupby("File_number", sort=True)[feature_cols]
    .mean()
)

assert len(surface_means) == 1000

real_means = real_X_geom.mean()
synth_global = synth_X.mean()

surface_means.to_csv(ARTIFACTS_DIR / "surface_feature_means.csv")
feature_summary = pd.DataFrame(
    {
        "feature": feature_cols,
        "synth_global_mean": [float(synth_global[c]) for c in feature_cols],
        "synth_global_std_triangles": [float(synth_X[c].std()) for c in feature_cols],
        "real_mean": [float(real_means[c]) for c in feature_cols],
        "real_std_triangles": [float(real_X_geom[c].std()) for c in feature_cols],
        "surface_means_mean": [float(surface_means[c].mean()) for c in feature_cols],
        "surface_means_std": [float(surface_means[c].std()) for c in feature_cols],
        "mean_abs_diff_real_vs_synth_global": [
            float(abs(real_means[c] - synth_global[c])) for c in feature_cols
        ],
        "std_abs_diff_real_vs_synth_global": [
            float(abs(real_X_geom[c].std() - synth_X[c].std())) for c in feature_cols
        ],
    }
)
feature_summary.to_csv(ARTIFACTS_DIR / "synth_vs_real_feature_summary.csv", index=False)
print("surfaces", len(surface_means))
feature_summary.head(8)


In [ ]:
def plot_surface_mean_violins(
    surface_means_df: pd.DataFrame,
    real_means_s: pd.Series,
    synth_global_s: pd.Series,
    cols: list[str],
    title: str,
    out_stem: str,
    figsize_scale: float = 1.0,
):
    """One violin per feature = distribution of per-surface means; strip = surfaces; markers = real + synth_global."""
    long = (
        surface_means_df[cols]
        .reset_index()
        .melt(id_vars=["File_number"], value_vars=cols, var_name="feature", value_name="value")
        .dropna(subset=["value"])
    )
    fig_w = max(6.0, 1.35 * len(cols)) * figsize_scale
    fig, ax = plt.subplots(figsize=(fig_w, 4.2))
    sns.violinplot(
        data=long,
        x="feature",
        y="value",
        ax=ax,
        inner=None,
        cut=0,
        color=COLOR_SURFACE,
        saturation=0.75,
    )
    sns.stripplot(
        data=long,
        x="feature",
        y="value",
        ax=ax,
        jitter=True,
        size=2.2,
        alpha=0.35,
        color="0.15",
        label="synth surface mean",
    )
    # highlighted markers at categorical positions
    xpos = {feat: i for i, feat in enumerate(cols)}
    ax.scatter(
        [xpos[c] for c in cols],
        [float(synth_global_s[c]) for c in cols],
        s=70,
        marker="s",
        color=COLOR_SYNTH_GLOBAL,
        zorder=5,
        label="synth_global mean",
    )
    ax.scatter(
        [xpos[c] for c in cols],
        [float(real_means_s[c]) for c in cols],
        s=80,
        marker="D",
        color=COLOR_REAL,
        zorder=6,
        label="real mean",
    )
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=35)
    # dedupe legend
    handles, labels = ax.get_legend_handles_labels()
    uniq = dict(zip(labels, handles))
    ax.legend(uniq.values(), uniq.keys(), loc="best", fontsize=8)
    fig.tight_layout()
    out = ARTIFACTS_DIR / f"{out_stem}.svg"
    fig.savefig(out, format="svg", dpi=1200, bbox_inches="tight")
    print("Wrote", out.resolve())
    plt.show()


batch_size = 6
for batch_i, start in enumerate(range(0, len(feature_cols), batch_size)):
    cols_batch = feature_cols[start : start + batch_size]
    plot_surface_mean_violins(
        surface_means,
        real_means,
        synth_global,
        cols_batch,
        title=f"Per-surface means - modeling features batch {batch_i + 1}",
        out_stem=f"surface_means_violin_strip_batch_{batch_i + 1}",
    )


## Relational features - Min / Intermediate / Max (surface means)

In [ ]:
relational_groups = {
    "Euclidean_N": ["Euclidean_N_Min", "Euclidean_N_Intermediate", "Euclidean_N_Max"],
    "Euclidean_D": ["Euclidean_D_Min", "Euclidean_D_Intermediate", "Euclidean_D_Max"],
    "Angle_N": ["Angle_N_Min", "Angle_N_Intermediate", "Angle_N_Max"],
    "Angle_D": ["Angle_D_Min", "Angle_D_Intermediate", "Angle_D_Max"],
    "Cosine_N": ["Cosine_N_Min", "Cosine_N_Intermediate", "Cosine_N_Max"],
    "Cosine_D": ["Cosine_D_Min", "Cosine_D_Intermediate", "Cosine_D_Max"],
}
rank_order = ["Min", "Intermediate", "Max"]

for group_name, cols in relational_groups.items():
    rename = {cols[0]: "Min", cols[1]: "Intermediate", cols[2]: "Max"}
    sm = surface_means[cols].rename(columns=rename)
    long = sm.reset_index().melt(
        id_vars=["File_number"], value_vars=rank_order, var_name="rank", value_name="value"
    )
    long["rank"] = pd.Categorical(long["rank"], categories=rank_order, ordered=True)

    fig, ax = plt.subplots(figsize=(7.5, 4.0))
    sns.violinplot(
        data=long, x="rank", y="value", ax=ax, inner=None, cut=0, color=COLOR_SURFACE
    )
    sns.stripplot(
        data=long,
        x="rank",
        y="value",
        ax=ax,
        jitter=True,
        size=2.2,
        alpha=0.35,
        color="0.15",
        label="synth surface mean",
    )
    xpos = {r: i for i, r in enumerate(rank_order)}
    ax.scatter(
        [xpos[rename[c]] for c in cols],
        [float(synth_global[c]) for c in cols],
        s=70,
        marker="s",
        color=COLOR_SYNTH_GLOBAL,
        zorder=5,
        label="synth_global mean",
    )
    ax.scatter(
        [xpos[rename[c]] for c in cols],
        [float(real_means[c]) for c in cols],
        s=80,
        marker="D",
        color=COLOR_REAL,
        zorder=6,
        label="real mean",
    )
    ax.set_title(f"{group_name}: per-surface Min / Intermediate / Max")
    handles, labels = ax.get_legend_handles_labels()
    uniq = dict(zip(labels, handles))
    ax.legend(uniq.values(), uniq.keys(), loc="best", fontsize=8)
    fig.tight_layout()
    out = ARTIFACTS_DIR / f"relational_surface_means_{group_name}.svg"
    fig.savefig(out, format="svg", dpi=1200, bbox_inches="tight")
    print("Wrote", out.resolve())
    plt.show()


## Fisher mean direction per surface (Fisher, Lewis & Embleton, 1993)

- Resultant vector R_vec = sum of unit vectors; R = ||R_vec||
- Mean direction = R_vec / R
- Mean resultant length R_bar = R / n
- Concentration approximation kappa ~= (n - 1) / (n - R) when R < n (descriptive; Fisher et al. 1993)

Computed for triangle **normals** `(X_N,Y_N,Z_N)` and **dip vectors** `(X_D,Y_D,Z_D)`

In [ ]:
def _unit_rows(xyz: np.ndarray) -> np.ndarray:
    xyz = np.asarray(xyz, dtype=float)
    norms = np.linalg.norm(xyz, axis=1, keepdims=True)
    norms = np.where(norms > 0, norms, np.nan)
    return xyz / norms


def fisher_summary(xyz: np.ndarray) -> dict:
    u = _unit_rows(xyz)
    u = u[np.isfinite(u).all(axis=1)]
    n = int(len(u))
    if n == 0:
        return {
            "n": 0,
            "mean_x": np.nan,
            "mean_y": np.nan,
            "mean_z": np.nan,
            "R": np.nan,
            "R_bar": np.nan,
            "kappa_approx": np.nan,
        }
    R_vec = u.sum(axis=0)
    R = float(np.linalg.norm(R_vec))
    R_bar = R / n
    mean = R_vec / R if R > 0 else np.array([np.nan, np.nan, np.nan])
    kappa = (n - 1) / (n - R) if n > 1 and R < n else np.nan
    return {
        "n": n,
        "mean_x": float(mean[0]),
        "mean_y": float(mean[1]),
        "mean_z": float(mean[2]),
        "R": R,
        "R_bar": float(R_bar),
        "kappa_approx": float(kappa) if np.isfinite(kappa) else np.nan,
    }


def fisher_per_surface(df: pd.DataFrame, cols: list[str], group_col: str | None) -> pd.DataFrame:
    rows = []
    if group_col is None:
        stats = fisher_summary(df[cols].to_numpy())
        stats["surface_id"] = "real"
        rows.append(stats)
    else:
        for sid, g in df.groupby(group_col):
            stats = fisher_summary(g[cols].to_numpy())
            stats["surface_id"] = sid
            rows.append(stats)
    return pd.DataFrame(rows)


fisher_normals_synth = fisher_per_surface(sorted_df, ["X_N", "Y_N", "Z_N"], "File_number")
fisher_normals_real = fisher_per_surface(real_sorted_df, ["X_N", "Y_N", "Z_N"], None)
fisher_normals_global = pd.DataFrame(
    [{**fisher_summary(sorted_df[["X_N", "Y_N", "Z_N"]].to_numpy()), "surface_id": "synth_global"}]
)

fisher_dip_synth = fisher_per_surface(sorted_df, ["X_D", "Y_D", "Z_D"], "File_number")
fisher_dip_real = fisher_per_surface(real_sorted_df, ["X_D", "Y_D", "Z_D"], None)
fisher_dip_global = pd.DataFrame(
    [{**fisher_summary(sorted_df[["X_D", "Y_D", "Z_D"]].to_numpy()), "surface_id": "synth_global"}]
)

fisher_normals_synth.to_csv(ARTIFACTS_DIR / "fisher_per_surface_normals.csv", index=False)
fisher_dip_synth.to_csv(ARTIFACTS_DIR / "fisher_per_surface_dip.csv", index=False)
fisher_normals_real.to_csv(ARTIFACTS_DIR / "fisher_real_normals.csv", index=False)
fisher_dip_real.to_csv(ARTIFACTS_DIR / "fisher_real_dip.csv", index=False)
fisher_normals_global.to_csv(ARTIFACTS_DIR / "fisher_synth_global_normals.csv", index=False)
fisher_dip_global.to_csv(ARTIFACTS_DIR / "fisher_synth_global_dip.csv", index=False)
print("Fisher normals surfaces", len(fisher_normals_synth))
fisher_normals_real


In [ ]:
def plot_fisher_surface_violins(synth_df, real_df, global_df, title, stem):
    metrics = ["mean_x", "mean_y", "mean_z", "R_bar", "kappa_approx"]
    fig, axes = plt.subplots(1, len(metrics), figsize=(14, 4.2), sharey=False)
    for ax, metric in zip(axes, metrics):
        vals = synth_df[metric].dropna()
        plot_df = pd.DataFrame({"metric": metric, "value": vals})
        label_synth = "synth surface" if ax is axes[0] else None
        label_global = "synth_global" if ax is axes[0] else None
        label_real = "real" if ax is axes[0] else None
        sns.violinplot(
            data=plot_df,
            x="metric",
            y="value",
            ax=ax,
            inner=None,
            cut=0,
            color=COLOR_SURFACE,
        )
        sns.stripplot(
            data=plot_df,
            x="metric",
            y="value",
            ax=ax,
            jitter=True,
            size=2.0,
            alpha=0.3,
            color="0.15",
            label=label_synth,
        )
        g_val = float(global_df[metric].iloc[0])
        r_val = float(real_df[metric].iloc[0])
        ax.scatter(
            [0],
            [g_val],
            s=70,
            marker="s",
            color=COLOR_SYNTH_GLOBAL,
            zorder=5,
            label=label_global,
        )
        ax.scatter(
            [0],
            [r_val],
            s=80,
            marker="D",
            color=COLOR_REAL,
            zorder=6,
            label=label_real,
        )
        y_all = np.concatenate([vals.to_numpy(dtype=float), [g_val, r_val]])
        y_all = y_all[np.isfinite(y_all)]
        if len(y_all):
            y_min, y_max = float(np.min(y_all)), float(np.max(y_all))
            pad = 0.05 * (y_max - y_min) if y_max > y_min else 0.05 * max(abs(y_max), 1.0)
            ax.set_ylim(y_min - pad, y_max + pad)
        ax.set_xlabel("")
        ax.set_ylabel(metric)
        ax.set_xticklabels([])
        ax.tick_params(axis="x", length=0)
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=3,
        fontsize=8,
        frameon=False,
    )
    fig.suptitle(title, y=1.02)
    fig.tight_layout(rect=[0, 0.08, 1, 0.96])
    out = ARTIFACTS_DIR / f"{stem}.svg"
    fig.savefig(out, format="svg", dpi=1200, bbox_inches="tight")
    print("Wrote", out.resolve())
    plt.show()


plot_fisher_surface_violins(
    fisher_normals_synth,
    fisher_normals_real,
    fisher_normals_global,
    "Fisher (normals) — per-surface distribution vs real / synth_global",
    "fisher_normals_surface_violin_strip",
)
plot_fisher_surface_violins(
    fisher_dip_synth,
    fisher_dip_real,
    fisher_dip_global,
    "Fisher (dip vectors) — per-surface distribution vs real / synth_global",
    "fisher_dip_surface_violin_strip",
)
